# WSD 学习率调度器

源码导航：[`core/utils/walkie_schedule.py`](../../../core/utils/walkie_schedule.py) 中的 `WalkieWSDSchedule`。

余弦退火（Cosine Annealing）与线性 warmup 的组合在中等规模训练中已被广泛使用，但其退火阶段难以提前停止并继续训练（因为重启后 lr 无法平滑衔接）。MiniCPM / Llama-3 等工作引入了 **WSD（Warmup-Stable-Decay）** 调度：将训练分为三段，Decay 段可被触发为独立的
，切换高质量小语料并平滑降至 final_lr，而不影响主训练阶段的连续性。

Walkie 进一步将两阶段数据（main + anneal）与 WSD 调度绑定在同一个 step 计数器上，AdamW 与 Muon 共享调度形状但各有独立的 peak_lr / final_lr 配置。

### 1. 理论推导

设全局训练步数为 $T$，warmup 步数为 $T_w$，退火开始步数为 $T_a = \lfloor T \times r_a \rfloor$（$r_a$ 为 `anneal_start_ratio`）。WSD 的学习率轨迹为：

$$
\eta(t) = \begin{cases}
  \eta_{\text{peak}} \cdot \dfrac{t}{T_w} & 0 \leq t < T_w \\
  \eta_{\text{peak}} & T_w \leq t < T_a \\
  \eta_{\text{final}} + (\eta_{\text{peak}} - \eta_{\text{final}}) \cdot f\!\left(\dfrac{t - T_a}{T - T_a}\right) & T_a \leq t \leq T
\end{cases}
$$

其中衰减形状函数 $f(p)$（$p \in [0,1]$）由 `decay_shape` 参数控制：

| `decay_shape` | $f(p)$ | 特点 |
|---|---|---|
| `sqrt` | $\sqrt{1-p}$ | 前期缓慢下降，后期快速归零，默认选项 |
| `linear` | $1-p$ | 匀速线性衰减 |
| `cosine` | $\frac{1}{2}(1 + \cos(\pi p))$ | 前后期均平滑，中段下降最快 |

注意在 $t = T_a$ 时 $f(0) = 1$，因此 $\eta(T_a) = \eta_{\text{peak}}$，**切换至退火阶段不产生 lr 跳变**（连续性保证）。

### 2. 多 optimizer 双轨配置

Walkie 的双优化器（AdamW 与 Muon）各自有独立的 `peak_lr` 和 `final_lr`。调度器内部以 `tracks: dict[str, _LRTrack]` 存储，`lrs_at(step)` 返回每个 track 的 lr 字典，供训练循环逐一写入优化器参数组。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.utils.walkie_schedule import WalkieWSDSchedule

### 3. 三种衰减形状可视化

In [ ]:
TOTAL = 1000
WARMUP = 100
ANNEAL_RATIO = 0.8

steps = list(range(TOTAL + 1))

fig, ax = plt.subplots(figsize=(10, 5))

for shape, color in [('sqrt', 'steelblue'), ('linear', 'darkorange'), ('cosine', 'green')]:
    sched = WalkieWSDSchedule(
        total_steps=TOTAL,
        warmup_steps=WARMUP,
        anneal_start_ratio=ANNEAL_RATIO,
        decay_shape=shape,
        tracks={'opt': type('T', (), {'peak_lr': 3e-4, 'final_lr': 3e-5})()},
    )
    lrs = [sched.lr_at(t, 'opt') for t in steps]
    ax.plot(steps, lrs, label=f'decay_shape={shape}', color=color, linewidth=2)

# 标注三个阶段的边界
ax.axvline(WARMUP,               color='gray', linestyle='--', alpha=0.7, label=f'warmup_end={WARMUP}')
ax.axvline(int(TOTAL*ANNEAL_RATIO), color='gray', linestyle=':',  alpha=0.7, label=f'anneal_start={int(TOTAL*ANNEAL_RATIO)}')

ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('WSD Schedule：三种衰减形状对比')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4. 调度器接口验证

In [ ]:
sched = WalkieWSDSchedule(
    total_steps=1000,
    warmup_steps=100,
    anneal_start_ratio=0.8,
    decay_shape='sqrt',
    tracks={
        'adamw': type('T', (), {'peak_lr': 3e-4, 'final_lr': 3e-5})(),
        'muon':  type('T', (), {'peak_lr': 2e-2, 'final_lr': 2e-3})(),
    },
)

# 关键边界点验证
print(f"step=0    lr(adamw)={sched.lr_at(0,'adamw'):.2e}  (应为 0)")
print(f"step=100  lr(adamw)={sched.lr_at(100,'adamw'):.2e}  (应为 peak_lr=3e-4)")
print(f"step=800  lr(adamw)={sched.lr_at(800,'adamw'):.2e}  (退火开始，应≈3e-4)")
print(f"step=1000 lr(adamw)={sched.lr_at(1000,'adamw'):.2e}  (应接近 final_lr=3e-5)")

# 验证退火前后 lr 连续（无跳变）
anneal_start = sched.anneal_start
diff = abs(sched.lr_at(anneal_start - 1, 'adamw') - sched.lr_at(anneal_start, 'adamw'))
print(f"\nanneal_start 前后 lr 差值: {diff:.2e}  (应接近 0）")

print(f"\nstage at step=500: {sched.current_stage(500)}")
print(f"stage at step=850: {sched.current_stage(850)}")

### 5. 源码精讲

```python
class WalkieWSDSchedule:
    total_steps: int
    warmup_steps: int
    anneal_start_ratio: float  # 退火开始的比例位置，默认 0.8
    decay_shape: str           # 'sqrt' | 'linear' | 'cosine'
    tracks: dict[str, _LRTrack]  # 每个 optimizer 的独立 peak_lr / final_lr

    @property
    def anneal_start(self) -> int:
        return int(self.total_steps * self.anneal_start_ratio)  # T_a

    def _shape_factor(self, step: int) -> float:
        """返回 [0, 1] 的衰减系数 f(p)。"""
        if step < self.warmup_steps:   return step / max(1, self.warmup_steps)
        if step < self.anneal_start:   return 1.0   # stable 段恒为 1
        progress = (step - anneal_start) / (total_steps - anneal_start)
        if self.decay_shape == 'sqrt':   return math.sqrt(1.0 - progress)
        if self.decay_shape == 'linear': return 1.0 - progress
        return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine

    def lr_at(self, step: int, name: str) -> float:
        track = self.tracks[name]
        f = self._shape_factor(step)
        # 线性插值：f=1 时 → peak_lr；f=0 时 → final_lr
        return track.final_lr + (track.peak_lr - track.final_lr) * f
```

训练循环中的典型用法：
```python
lrs = schedule.advance()  # step += 1，返回各 track 的最新 lr
for name, lr in lrs.items():
    for pg in optimizers[name].param_groups:
        pg['lr'] = lr
```

---

## 延伸阅读与参考资料

### 论文与博客
- **MiniCPM（WSD 调度首次系统应用）**: Hu et al., 2024. [arXiv:2404.06395](https://arxiv.org/abs/2404.06395)
- **Cyclical Learning Rates**: Smith, 2017. [arXiv:1506.01186](https://arxiv.org/abs/1506.01186)

### 工程实现
- **PyTorch LambdaLR / get_scheduler**: [docs](https://pytorch.org/docs/stable/optim.html#torch.optim.lr_scheduler.LambdaLR)
- **Hugging Face get_wsd_schedule**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/optimization.py)